# Phase 1 — EDA: Towards Comprehensive Dataset

**Goal:** Load and merge the training data, then explore its structure before building models.

| Step | What we do |
|------|------------|
| 1 | Merge `Dataset.csv` + `Repetition.csv` on `(User1, User2)` |
| 2 | Inspect shape, columns, and class balance |
| 3 | Save merged file for reuse |

**Dataset:** `datasets/towards-comprehensive/`  
**Output:** `outputs/data/merged_towards_comprehensive.csv`

## 1. Load and merge data

Each row is one **message**. We join **Repetition.csv** so every message gets relational features for its user pair:

| Column | Source | Meaning |
|--------|--------|---------|
| `Message` | Dataset.csv | Text to classify |
| `CBLabel` | Dataset.csv | Target: **0** = non-cyberbullying, **1** = cyberbullying |
| `SubLabel` | Dataset.csv | 1 = aggressive, 2 = non-aggressive |
| `User1`, `User2` | Both | User pair (merge key) |
| `TotalMsgs` | Repetition.csv | Total messages between this pair |
| `AggressiveMsgs` | Repetition.csv | Aggressive messages between this pair |
| `Peerness` | Repetition.csv | Peer relationship score (0–1) |

In [1]:
import sys
from pathlib import Path

# Find F21MP project root (works whether cwd is project root or notebooks/phase1_training/)
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Cannot find F21MP project root. Open the F21MP folder in Cursor/VS Code, "
        "or run Jupyter with cwd set to the project root."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase1_training.data.load_towards_comprehensive import load_and_merge, DEFAULT_MERGED_OUTPUT
import pandas as pd

df = load_and_merge(save_path=DEFAULT_MERGED_OUTPUT)
print(df.shape)
print(df.columns.tolist())
if "CBLabel" in df.columns:
    print(df["CBLabel"].value_counts())
print("Merged data saved to:", DEFAULT_MERGED_OUTPUT)

(15554, 10)
['Date', 'Time', 'User1', 'User2', 'Message', 'SubLabel', 'CBLabel', 'TotalMsgs', 'AggressiveMsgs', 'Peerness']
CBLabel
0    15428
1      126
Name: count, dtype: int64
Merged data saved to: /Users/pharshitha/Desktop/F21MP/outputs/data/merged_towards_comprehensive.csv


## 2. EDA results — what to expect

After running the cell above, you should see something like:

| Metric | Expected value | Meaning |
|--------|----------------|---------|
| **Shape** | `(15554, 10)` | ~15.5k messages, 10 columns after merge |
| **CBLabel = 0** | ~15,428 | Non-cyberbullying (majority class) |
| **CBLabel = 1** | ~126 | Cyberbullying (minority class) |
| **Imbalance ratio** | ~99 : 1 | Heavily skewed — **do not rely on accuracy alone** |

### Key takeaway

The dataset is **extremely imbalanced** (~0.8% positive). Phase 1 training therefore uses:
- `scale_pos_weight` in XGBoost to penalise missed cyberbullying cases
- **F1, precision, recall** as primary metrics (not accuracy)

### Minor note

~53 rows may be dropped during training if tabular features are missing — final training set is typically **~15,501** rows.

In [2]:
# Quick exploration — class balance and sample preview
import pandas as pd

if "CBLabel" in df.columns:
    counts = df["CBLabel"].value_counts()
    pct = (counts / len(df) * 100).round(2)
    print("Class balance:")
    print(pd.DataFrame({"count": counts, "percent": pct}))

print("\nFirst 3 rows (key columns):")
cols = ["User1", "User2", "Message", "CBLabel", "Peerness", "TotalMsgs", "AggressiveMsgs"]
print(df[cols].head(3).to_string())

print("\nPeerness & repetition stats:")
print(df[["Peerness", "TotalMsgs", "AggressiveMsgs"]].describe().round(2).to_string())

Class balance:
         count  percent
CBLabel                
0        15428    99.19
1          126     0.81

First 3 rows (key columns):
   User1  User2                                                                                                                                            Message  CBLabel  Peerness  TotalMsgs  AggressiveMsgs
0    194     89  The last few days I have been working in this article. I think it is now optimal. Could you take a close look at it and rephrase where necessary.        0  0.672222        1.0             0.0
1    139    109                                                           Just another white supremacist dog. 'nuff said.  Only a motherfucker shoots prisoners.          0  0.433333        2.0             1.0
2    175    125                                                                                                                          Support. Same reason as.         0  0.277778        1.0             0.0

Peerness & repetition s

## 3. Phase 1 EDA — complete ✓

**Done:**
- Data loaded and merged
- Class imbalance documented
- Merged CSV saved to `outputs/data/`

**Next:** Run `02_training.ipynb` to train baseline (A1) and full model (A4).